# Finetune DNABERT2

In [2]:
import os
import pandas as pd

## Clone DNABERT2 repository

In [1]:
!git clone https://github.com/MAGICS-LAB/DNABERT_2.git

Cloning into 'DNABERT_2'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 126 (delta 31), reused 20 (delta 20), pack-reused 82 (from 1)
Receiving objects: 100% (126/126), 879.78 KiB | 16.29 MiB/s, done.
Resolving deltas: 100% (55/55), done.


## Create dataset

In [3]:
def load_perphect_sequences(data_dir: str):
    bact = pd.read_csv(os.path.join(data_dir, "bacteria_df.csv"))["bacterium_sequence"].tolist()
    phag = pd.read_csv(os.path.join(data_dir, "phages_df.csv"))["phage_sequence"].tolist()

    return bact, phag

bact_sequences, phag_sequences = load_perphect_sequences("../data/perphect-data/all/")

In [15]:
def save_dataset(seqs, output_dir):
    total = len(seqs)

    train_end = int(0.8 * total)
    val_end = int(0.9 * total)

    datasets = {
        "train": seqs[:train_end],
        "dev": seqs[train_end:val_end],
        "test": seqs[val_end:]
    }

    os.makedirs(output_dir, exist_ok=True)

    for i, (split, data) in enumerate(datasets.items()):
        with open(os.path.join(output_dir, f"{split}.csv"), "w") as f:
            for seq in data:
                f.write(f"{seq[:2048]},{i}\n")

In [16]:
save_dataset(phag_sequences, "./datasets/dnabert_phages/")

In [8]:
%cd DNABERT_2/finetune/

/data/pere.carrillo/pbi/finetuning/DNABERT_2/finetune


/home/pere.carrillo/micromamba/envs/pbi-dnabert/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [17]:
DATA_PATH="../../datasets/dnabert_phages"  # e.g., ./sample_data
MAX_LENGTH=100 # Please set the number as 0.25 * your sequence length. 
											# e.g., set it as 250 if your DNA sequences have 1000 nucleotide bases
											# This is because the tokenized will reduce the sequence length by about 5 times
LR=3e-5
!export CUDA_VISIBLE_DEVICES=2  # Set GPU IDs

# Training use DataParallel
!python train.py \
    --model_name_or_path zhihan1996/DNABERT-2-117M \
    --data_path  {DATA_PATH} \
    --kmer -1 \
    --run_name DNABERT2_{DATA_PATH} \
    --model_max_length {MAX_LENGTH} \
    --per_device_train_batch_size 8 \
    --per_device_eval_batch_size 16 \
    --gradient_accumulation_steps 1 \
    --learning_rate {LR} \
    --num_train_epochs 5 \
    --fp16 \
    --save_steps 200 \
    --output_dir output/dnabert2 \
    --evaluation_strategy steps \
    --eval_strategy steps \
    --save_strategy steps \
    --eval_steps 200 \
    --warmup_steps 50 \
    --logging_steps 100 \
    --overwrite_output_dir True \
    --log_level info \
    --find_unused_parameters False

2025-12-01 17:12:21.883618: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-01 17:12:21.922848: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-01 17:12:21.922891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-01 17:12:21.923943: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-01 17:12:21.930108: I tensorflow/core/platform/cpu_feature_guar